# Testing the `src` Statistical Package

**Advanced Examples**

This notebook demonstrates advanced usage of the core reusable modules in `src/`.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import bootstrap
from src import hypothesis_testing
from src import empirical_bayes
from src import shrinkage
from src import lasso
from src import survival

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

## 1. Bootstrap: Advanced Confidence Intervals

Compare **percentile** vs **basic** bootstrap methods and analytical CI.

In [ ]:
data = np.random.normal(loc=100, scale=20, size=200)

def mean_stat(x): return np.mean(x)

ci_percentile = bootstrap.bootstrap_confidence_interval(data, mean_stat, method='percentile')
ci_basic      = bootstrap.bootstrap_confidence_interval(data, mean_stat, method='basic')

analytical_se = np.std(data, ddof=1) / np.sqrt(len(data))
analytical_ci = (np.mean(data) - 1.96*analytical_se, np.mean(data) + 1.96*analytical_se)

print("Analytical 95% CI     :", np.round(analytical_ci, 2))
print("Bootstrap Percentile  :", np.round(ci_percentile, 2))
print("Bootstrap Basic       :", np.round(ci_basic, 2))

---

## 2. Hypothesis Testing: Multiple Tests + Permutation

In [ ]:
# One-sample t-test
sample = np.random.normal(50, 10, 80)
t, p = hypothesis_testing.one_sample_t_test(sample, popmean=48)
print(f"One-sample t-test: t={t:.3f}, p={p:.4f}")

# Chi-square test (independence)
observed = np.array([[50, 30], [20, 40]])
chi2, p_chi, dof = hypothesis_testing.chi_square_test(observed)
print(f"Chi-square test: chi2={chi2:.2f}, p={p_chi:.4f}, dof={dof}")

# Permutation test between two groups
group_a = np.random.normal(120, 15, 60)
group_b = np.random.normal(130, 18, 55)
obs, p_perm = hypothesis_testing.permutation_test(group_a, group_b, n_permutations=5000)
print(f"Permutation test diff: {obs:.2f}, p={p_perm:.4f}")

---

## 3. Empirical Bayes & Shrinkage: Before vs After Visualization

In [ ]:
true_means = np.array([45, 52, 38, 61, 47, 55, 40])
observations = true_means + np.random.normal(0, 8, len(true_means))

shrunk_js = empirical_bayes.james_stein_estimator(observations)
shrunk_pp = shrinkage.positive_part_james_stein(observations)

plt.figure(figsize=(10,5))
x = np.arange(len(observations))
plt.plot(x, observations, 'o-', label='Raw Observations', alpha=0.7)
plt.plot(x, shrunk_js, 's--', label='James-Stein', linewidth=2)
plt.plot(x, shrunk_pp, '^-.', label='Positive-Part JS', linewidth=2)
plt.axhline(np.mean(observations), color='gray', linestyle=':', label='Grand Mean')
plt.legend()
plt.title('Shrinkage Effect: Raw vs Shrunk Estimates')
plt.xlabel('Group')
plt.ylabel('Estimate')
plt.show()

print("Shrinkage reduces variance while preserving signal.")

---

## 4. LASSO: Feature Selection on Simulated Data

In [ ]:
n, p = 150, 10
X = np.random.randn(n, p)
true_beta = np.array([4, 0, -3, 0, 2.5, 0, 0, 1.8, 0, -1.2])
y = X @ true_beta + np.random.normal(0, 2, n)

coef, model = lasso.fit_lasso_sklearn(X, y, alpha=0.8)

plt.figure(figsize=(8,4))
plt.bar(range(p), coef)
plt.title('LASSO Coefficients (alpha=0.8)')
plt.xlabel('Feature Index')
plt.ylabel('Coefficient Value')
plt.show()

print("Non-zero coefficients (selected features):", np.where(np.abs(coef) > 1e-5)[0])

---

## 5. Survival Analysis: Kaplan-Meier + Log-Rank Test

In [ ]:
# Simulate two groups
np.random.seed(42)
dur_A = np.random.exponential(12, 80)
event_A = np.random.binomial(1, 0.65, 80)

dur_B = np.random.exponential(18, 70)
event_B = np.random.binomial(1, 0.55, 70)

km_A = survival.kaplan_meier_estimator(dur_A, event_A)
km_B = survival.kaplan_meier_estimator(dur_B, event_B)

plt.figure(figsize=(8,5))
plt.step(km_A['timeline'], km_A['KM_estimate'], where='post', label='Group A')
plt.step(km_B['timeline'], km_B['KM_estimate'], where='post', label='Group B')
plt.xlabel('Time')
plt.ylabel('Survival Probability')
plt.title('Kaplan-Meier Curves')
plt.legend()
plt.show()

stat, p_logrank = survival.log_rank_test(dur_A, event_A, dur_B, event_B)
print(f"Log-rank test statistic: {stat:.2f}, p-value: {p_logrank:.4f}")

---

## Summary

All core modules in `src/` are working correctly with both basic and advanced usage.

You can now confidently use these modules in your real projects and examples.